Imports

In [1]:
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import numpy as np
import ee

In [2]:
ee.Authenticate()
ee.Initialize(project='thesis-phenology')
print("Earth Engine API initialized successfully.")



Successfully saved authorization token.
Earth Engine API initialized successfully.


Load coordinates

In [3]:
output_dir = Path("output")
# Find subfolders in output folder to choose input

subfolders = sorted([f for f in output_dir.iterdir() if f.is_dir()])

if not subfolders:
    raise FileNotFoundError(f"No subdirectories found inside '{output_dir}'.")

if len(subfolders) == 1:
    selected_folder = subfolders[0]
    print(f"Only one folder found. Automatically selected: {selected_folder.name}.")

else:
    print("Available folders in output directory:")
    for idx, folder in enumerate(subfolders, start=1):
        print(f"[{idx}] {folder.name}")

# Get user input
    user_choice = input(f"\nEnter folder number (1-{len(subfolders)}): ").strip()

    if not user_choice.isdigit() or not (1 <= int(user_choice) <= len(subfolders)):
        raise ValueError(
            f"Invalid selection '{user_choice}'. Expected number between 1 and {len(subfolders)}."
        )

    selected_folder = subfolders[int(user_choice) - 1]
    print(f"\n --> Selected folder: {selected_folder.name}")


# Locate modeling file
modeling_files = sorted(selected_folder.glob("*_modeling*.csv"))

# Pick the file
modeling_file = modeling_files[0]

if not modeling_file:
    raise FileNotFoundError(f"No '*_modeling_*.csv' file found in {selected_folder.name}.")


print(f"Loading coordinates from: {modeling_file}")

df_master = pd.read_csv(modeling_file, sep=";")
df_master["Year"] = df_master["Year"].astype(int)

# Filter to 2017 onwards (Sentinel-2 Surface Reflectance availability)
s2_targets = (
    df_master[df_master["Year"] >= 2017][["Latitude", "Longitude", "Year"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Total unique station-years to query from Sentinel-2 (2017+): {len(s2_targets)}")
display(s2_targets.head())

Available folders in output directory:
[1] france_sbt_bbch39_2017-01-01_2022-12-31
[2] germany_barw_bbch51_2017-01-01_2025-12-31
[3] germany_whbw_bbch0_2017-01-01_2020-12-31

 --> Selected folder: germany_barw_bbch51_2017-01-01_2025-12-31
Loading coordinates from: output\germany_barw_bbch51_2017-01-01_2025-12-31\curated_for_modeling.csv
Total unique station-years to query from Sentinel-2 (2017+): 4942


,Latitude,Longitude,Year
0,47.7333,9.06667,2017
1,47.7333,9.06667,2018
2,47.7333,9.06667,2019
3,47.7333,9.06667,2020
4,47.7333,9.06667,2021


Sentinel-2 Band Extraction (B4, B8, SCL)

In [4]:
raw_dir = Path("data")
s2_raw_dir = Path(raw_dir / f"s2_ndvi_raw_{selected_folder.name}")
s2_raw_dir.mkdir(parents=True, exist_ok=True)

# Filter out station-years that are already downloaded on disk
pending_s2 = []
for idx, row in s2_targets.iterrows():
    lat = float(row["Latitude"])
    lon = float(row["Longitude"])
    year = int(row["Year"])
    out_csv = s2_raw_dir / f"s2_{idx:04d}_{year}_{lat:.4f}_{lon:.4f}.csv"
    
    if not (out_csv.exists() and out_csv.stat().st_size > 50):
        pending_s2.append((idx, lat, lon, year, out_csv))

print(f"Total targets: {len(s2_targets)} | Already cached: {len(s2_targets) - len(pending_s2)} | Remaining: {len(pending_s2)}")

# Submit GEE getRegion calls in parallel
executor = ThreadPoolExecutor(max_workers=8)
future_to_meta = {}

for idx, lat, lon, year, out_csv in pending_s2:
    pt = ee.Geometry.Point([lon, lat])
    ic = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(pt)
        .filterDate(f"{year}-01-01", f"{year}-12-31")
        .select(["B4", "B8", "SCL"])
    )
    region_call = ic.getRegion(pt, 10)
    
    # Submit bound method .getInfo directly (no custom function needed)
    future = executor.submit(region_call.getInfo)
    future_to_meta[future] = (idx, lat, lon, year, out_csv)

completed_count = 0
for future in as_completed(future_to_meta):
    idx, lat, lon, year, out_csv = future_to_meta[future]
    completed_count += 1
    try:
        rows = future.result()
        if len(rows) > 1:
            part_df = pd.DataFrame(rows[1:], columns=rows[0])
            part_df["Station_Latitude"] = lat
            part_df["Station_Longitude"] = lon
            part_df["Year"] = year
            part_df.to_csv(out_csv, sep=";", index=False)
        if completed_count % 50 == 0 or completed_count == len(pending_s2):
            print(f"Completed {completed_count}/{len(pending_s2)} GEE extractions...")
    except Exception as e:
        print(f"Error on ({lat:.4f}, {lon:.4f}) in {year}: {e}")

executor.shutdown(wait=True)
print("\nSentinel-2 extraction complete!")

Total targets: 4942 | Already cached: 0 | Remaining: 4942
Completed 50/4942 GEE extractions...
Completed 100/4942 GEE extractions...
Completed 150/4942 GEE extractions...
Completed 200/4942 GEE extractions...
Completed 250/4942 GEE extractions...
Completed 300/4942 GEE extractions...
Completed 350/4942 GEE extractions...
Completed 400/4942 GEE extractions...
Completed 450/4942 GEE extractions...
Completed 500/4942 GEE extractions...
Completed 550/4942 GEE extractions...
Completed 600/4942 GEE extractions...
Completed 650/4942 GEE extractions...
Completed 700/4942 GEE extractions...
Completed 750/4942 GEE extractions...
Completed 800/4942 GEE extractions...
Completed 850/4942 GEE extractions...
Completed 900/4942 GEE extractions...
Completed 950/4942 GEE extractions...
Completed 1000/4942 GEE extractions...
Completed 1050/4942 GEE extractions...
Completed 1100/4942 GEE extractions...
Completed 1150/4942 GEE extractions...
Completed 1200/4942 GEE extractions...
Completed 1250/4942 GEE ex

Cloud Masking, NDVI Calculation, Daily Interpolation & Master File Merge

In [5]:
s2_files = sorted(s2_raw_dir.glob("s2_*.csv"))

if not s2_files:
    raise FileNotFoundError(f"No Sentinel-2 CSVs found in {s2_raw_dir.resolve()}.")

s2_df = pd.concat([pd.read_csv(f, sep=";") for f in s2_files], ignore_index=True)

# 1. Convert GEE millisecond timestamp to datetime and DOY
s2_df["valid_time"] = pd.to_datetime(s2_df["time"], unit="ms", errors="coerce")
s2_df["S2_DOY"] = s2_df["valid_time"].dt.dayofyear

# 2. Cloud masking via Scene Classification Layer (SCL)
# Keep: 4 (Vegetation), 5 (Bare soil), 6 (Water), 7 (Unclassified low-cloud)
# Exclude: 1, 2, 3 (Shadows), 8, 9, 10 (Clouds/Cirrus), 11 (Snow)
s2_clean = s2_df[s2_df["SCL"].isin([4, 5, 6, 7])].copy()

# 3. Compute NDVI = (B8 - B4) / (B8 + B4)
b8 = pd.to_numeric(s2_clean["B8"], errors="coerce")
b4 = pd.to_numeric(s2_clean["B4"], errors="coerce")
s2_clean["NDVI"] = ((b8 - b4) / (b8 + b4)).clip(lower=-0.2, upper=1.0)

s2_clean = s2_clean.dropna(subset=["NDVI", "S2_DOY"])
s2_clean = s2_clean[s2_clean["S2_DOY"] <= 365].copy()

# 4. Pivot to wide format (1 row per station-year, columns = DOY 1..365)
ndvi_wide = s2_clean.pivot_table(
    index=["Station_Latitude", "Station_Longitude", "Year"],
    columns="S2_DOY",
    values="NDVI",
    aggfunc="mean",
)

# Ensure all 365 days exist as columns, then linearly interpolate cloudy gaps
ndvi_wide = ndvi_wide.reindex(columns=range(1, 366))
ndvi_wide = (
    ndvi_wide.interpolate(method="linear", axis=1)
    .bfill(axis=1)
    .ffill(axis=1)
)

ndvi_feature_cols = [f"NDVI_doy_{col}" for col in range(1, 366)]
ndvi_wide.columns = ndvi_feature_cols
ndvi_wide = ndvi_wide.reset_index().rename(
    columns={"Station_Latitude": "Latitude", "Station_Longitude": "Longitude"}
)

# 5. Merge NDVI_doy_1...NDVI_doy_365 into the master modeling file
existing_ndvi_cols = [c for c in df_master.columns if c.startswith("NDVI_doy_")]
if existing_ndvi_cols:
    df_master = df_master.drop(columns=existing_ndvi_cols)

df_master = df_master.merge(
    ndvi_wide,
    on=["Latitude", "Longitude", "Year"],
    how="left",
)

# Save updated master file
df_master.to_csv(modeling_file, sep=";", index=False, encoding="utf-8")

print(f"Updated master modeling file saved to: {modeling_file.resolve()}")
print(f"Rows with complete 365-day NDVI profiles: {df_master['NDVI_doy_180'].notna().sum()} / {len(df_master)}")
display(
    df_master[df_master["NDVI_doy_180"].notna()][
        ["Latitude", "Longitude", "Year", "DOY", "NDVI_doy_90", "NDVI_doy_150", "NDVI_doy_180", "NDVI_doy_210"]
    ].head()
)

Updated master modeling file saved to: C:\Users\jimpa\Desktop\Thesis\Notebooks\output\germany_barw_bbch51_2017-01-01_2025-12-31\curated_for_modeling.csv
Rows with complete 365-day NDVI profiles: 4942 / 4942


,Latitude,Longitude,Year,DOY,NDVI_doy_90,NDVI_doy_150,NDVI_doy_180,NDVI_doy_210
0,47.7333,9.06667,2017,140,0.716490,0.866456,0.726240,0.893205
1,47.7333,9.06667,2018,125,0.537974,0.506446,0.720671,0.540253
2,47.7333,9.06667,2019,138,0.691879,0.918435,0.466323,0.852324
3,47.7333,9.06667,2020,132,0.741732,0.835001,0.610911,0.842444
4,47.7333,9.06667,2021,142,0.794140,0.857250,0.718157,0.666425
